<a href="https://colab.research.google.com/github/marpablo/Marian_INFO4670_Fall2026/blob/main/info_4670_assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 - Cleaning & Integrating the Northgate Data

Name: Marian Pablo

Date: 09/20/2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
try:
    import pandas as pd
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Please upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")

print("students :", students.shape)
print("enroll   :", enroll.shape)
print("activity :", activity.shape)

students : (2027, 12)
enroll   : (8088, 4)
activity : (32000, 4)


# A1 - Missing values



In [ ]:
# Where are the blanks?
missing = students.isna().sum()
print(missing[missing > 0])
n_missing = students["study_hours_reported"].isna().sum()
print(f"\nstudy_hours_reported: {n_missing} blank of {len(students)} "
      f"({n_missing/len(students)*100:.1f}%) -> {students['study_hours_reported'].notna().sum()} present")

study_hours_reported    255
dtype: int64

study_hours_reported: 255 blank of 2027 (12.6%) -> 1772 present


In [ ]:
# study_hours is skewed, so Han method 4 uses the MEDIAN (not the mean).
n_missing_study = students["study_hours_reported"].isna().sum()
med = students["study_hours_reported"].median()
print("median study hours (skew -> median):", med)

# Two honest options on a COPY (never touch the raw column):
filled  = students["study_hours_reported"].fillna(med)     # method 4: impute median
flagged = students["study_hours_reported"].isna()          # or: keep a 'was missing' flag
print("after median fill, blanks remaining:", filled.isna().sum())
print("rows flagged as originally missing:", flagged.sum())

median study hours (skew -> median): 10.5
after median fill, blanks remaining: 0
rows flagged as originally missing: 255


In [ ]:
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


Justification: I went with the attribute's central tendency method. There is a skew so using the median would allow for the data to avoid further distortion. The sample size before was 1,772, and after the 255 blanks have been filled, the sample size is now 2,027.

# A2 - Inconsistent Categories

In [ ]:
print("RAW housing — eight spellings for three groups:")
print(students["housing"].value_counts())

clean_map = {"on-campus":"On-Campus", "off-campus":"Off-Campus",
             "off campus":"Off-Campus", "with family":"With Family"}
students["housing_clean"] = students["housing"].str.strip().str.lower().map(clean_map)

print("\nCLEANED — three real groups:")
print(students["housing_clean"].value_counts())

RAW housing — eight spellings for three groups:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

CLEANED — three real groups:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [ ]:
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


Value counts before and after

**RAW housing — eight spellings for three groups:**

housing
Off-Campus  -   755

On-Campus   -   480

With Family  -  420

off campus   -  113

Off-campus   -   77

with family  -   72


on-campus   -    69


 On-Campus   -   41



**CLEANED — three real groups:**

Off-Campus  -    945

On-Campus   -   590

With Family  -  492

# A3 - Errors vs. extremes

In [ ]:
# Impossible values break a rule you can write down -> they are errors.
imp_ages_calculated = sorted(int(a) for a in students.loc[(students["age"] < 15) | (students["age"] > 90), "age"].unique())
neg_work_calculated = (students["work_hours_per_week"] < 0).sum()
print("impossible ages:", imp_ages_calculated)
print("students with negative work hours:", neg_work_calculated)

# Assign the calculated values to the variables used in the assert
impossible_ages = imp_ages_calculated
n_neg_work = neg_work_calculated

# The 43 zero-GPAs are ambiguous, not impossible.
zeros = students["final_gpa"] == 0
print("\nGPA = 0.00 rows:", int(zeros.sum()),
      "| of those, marked dropped_out:", int(students.loc[zeros, "dropped_out"].sum()))

impossible ages: [-22, 0, 1, 3, 199, 220]
students with negative work hours: 4

GPA = 0.00 rows: 43 | of those, marked dropped_out: 9


In [ ]:
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


Impossible ages include -22,  199, and 220. It is not possible for someone to be a negative age. 199 & 220 isn't reasonable as it is too old. For these values, we should remove them. Any work hours less than 0 cannot be true. It can most often be a typo, so we can fix these values and use their absolute value. An impossible value differs from an extreme but valid value since you must think of it in a logical way and which can exist in the real world. For example, 60 may be old for a student but it is still very possible and should be kept as a valid age.

# A4 - Duplicate student records

In [ ]:
# student_records SHOULD be one row per student:
students_dedup = students.copy()
students_dedup = students_dedup.drop_duplicates() # Remove exact duplicate rows
students_dedup = students_dedup.drop_duplicates(subset='student_id') # Ensure one row per student_id
print('student_records rows:', len(students), '| unique student_id:', students['student_id'].nunique())
print('exact duplicate rows:', students.duplicated().sum(),
      '| duplicate student_ids:', students['student_id'].duplicated().sum())

# enroll / activity have MANY rows per student BY DESIGN -> not duplicates:
print('\nenroll : one row per enrollment ->', len(enroll), 'rows for', enroll['sid'].nunique(), 'students')
print('activity: one row per student-week ->', len(activity), 'rows for',
      activity['student_id'].nunique(), 'students x', activity['week'].nunique(), 'weeks')

student_records rows: 2027 | unique student_id: 2000
exact duplicate rows: 18 | duplicate student_ids: 27

enroll : one row per enrollment -> 8088 rows for 2014 students
activity: one row per student-week -> 32000 rows for 2000 students x 16 weeks


In [ ]:
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


When it comes to de-duplicating, course_enrollments and weekly_activities may have repeating data but it all  belongs to a unique student id. If we were to delete the duplicates, then a student id would not have their real transactional data.

# B5 - Standardize the key and integrate

In [ ]:
# build the standardized key and the one-row-per-student "analysis" table
print("student_records key:", students["student_id"].iloc[0], "->", students["student_id"].dtype)
print("course_enroll  key:", enroll["sid"].iloc[0], "->", enroll["sid"].dtype)

# (1) As-is: text vs number -> pandas refuses.
try:
    pd.merge(students, enroll, left_on="student_id", right_on="sid")
except Exception as e:
    print("\nJoin as-is FAILS:", type(e).__name__, "-", str(e)[:55])

# (2) Both as text: it runs, but nothing matches ("NU-101508" != "108933").
both = pd.merge(students.assign(k=students["student_id"].astype(str)),
                enroll.assign(k=enroll["sid"].astype(str)), on="k")
print("Both as text -> rows matched:", len(both))

student_records key: NU-101508 -> object
course_enroll  key: 108933 -> int64

Join as-is FAILS: ValueError - You are trying to merge on object and int64 columns for
Both as text -> rows matched: 0


In [ ]:
# (3) Standardize: strip "NU-", match on the number.
key = students["student_id"].str.replace("NU-", "", regex=False).astype(int)

matched      = enroll["sid"].isin(set(key)).sum()
orphan_sids  = set(enroll["sid"]) - set(key)
print("enrollment rows that match a student:", matched, "of", len(enroll))
print("orphan IDs (no student record):", len(orphan_sids),
      "-> orphan rows:", enroll["sid"].isin(orphan_sids).sum())

# The deferred check: do the 43 zero-GPA students actually exist elsewhere?
zero_keys = set(key[students["final_gpa"].values == 0])
act_keys  = set(activity["student_id"].str.replace("NU-", "", regex=False).astype(int))
print("\n43 zeros with an enrollment:", len(zero_keys & set(enroll["sid"])),
      "| with activity:", len(zero_keys & act_keys))

enrollment rows that match a student: 8041 of 8088
orphan IDs (no student record): 14 -> orphan rows: 47

43 zeros with an enrollment: 43 | with activity: 43


In [ ]:
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


# B6 - Verify the join

8041 of 8088 rows matched.

14 orphan IDs (47 rows) appeared.



In [ ]:
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


# C7 - Parse the dates

In [ ]:
import re
def date_format(s):
    if re.match(r"^\d{4}-\d{2}-\d{2}$", s):        return "YYYY-MM-DD"
    if re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", s):     return "M/D/YYYY"
    if re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$", s):  return "D-Mon-YYYY"
    return "other"
print(students["enrollment_date"].map(date_format).value_counts())

naive        = pd.to_datetime(students["enrollment_date"], errors="coerce")             # the trap
dates_parsed = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")
print("\nnaive parse      -> blanks (NaT):", naive.isna().sum())
print("dates_parsed -> blanks (NaT):", dates_parsed.isna().sum())

enrollment_date
M/D/YYYY      1165
YYYY-MM-DD     557
D-Mon-YYYY     305
Name: count, dtype: int64

naive parse      -> blanks (NaT): 1470
dates_parsed -> blanks (NaT): 0


In [ ]:
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


# C8 - Normalize and discretize

In [ ]:
# On our data: z-score the columns a distance-based method would use.
analysis = students.copy() # Create a new DataFrame for analysis
analysis["gpa_band"] = bands # Add the 'gpa_band' column using the pre-computed 'bands' Series

for col in ["commute_miles", "study_hours_reported"]:
    z = (students[col] - students[col].mean()) / students[col].std()
    # Add the z-score to the 'analysis' DataFrame
    if col == "study_hours_reported":
        analysis["study_z"] = z # Specifically named as 'study_z'
    else:
        analysis[f"{col}_z"] = z # Generic naming for other z-scored columns

    print(f"{col}: after z-score  mean = {z.mean():.2f},  sd = {z.std():.2f}")

commute_miles: after z-score  mean = -0.00,  sd = 1.00
study_hours_reported: after z-score  mean = 0.00,  sd = 1.00


In [ ]:
bands = pd.cut(students["final_gpa"], bins=[-0.01, 1, 2, 3, 4],
               labels=["0-1", "1-2", "2-3", "3-4"])
print(bands.value_counts().sort_index())

final_gpa
0-1      71
1-2     532
2-3    1118
3-4     306
Name: count, dtype: int64


In [ ]:
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


# D9 - Write the clean file

In [ ]:
import numpy as np # Ensure numpy is imported for np.nan
clean = students.copy()                                   # never touch the raw

# 1. de-duplicate (student_records only: one row per student)
clean = clean.drop_duplicates()                          # 18 exact copies
clean = clean.drop_duplicates(subset="student_id")       # 9 near-duplicates -> keep first

# 2. standardize categories (already computed as housing_clean)
# 3. fix impossible values -> mark missing
clean.loc[(clean["age"] < 15) | (clean["age"] > 90), "age"] = np.nan
clean.loc[clean["work_hours_per_week"] < 0, "work_hours_per_week"] = np.nan
# 4. parse dates deliberately
clean["enrollment_date"] = pd.to_datetime(clean["enrollment_date"], format="mixed", errors="coerce")
# 5. standardize the key for joining
clean["sid"] = clean["student_id"].str.replace("NU-", "", regex=False).astype(int)

print("clean student table:", clean.shape)
clean.to_csv("student_records_clean.csv", index=False)   # a NEW file; raw stays untouched
print("wrote student_records_clean.csv")

# Save the analysis DataFrame as well, as suggested by the original comment
analysis.to_csv("northgate_clean.csv", index=False)
print("wrote northgate_clean.csv")

clean student table: (2000, 14)
wrote student_records_clean.csv
wrote northgate_clean.csv


# D10 - Cleaning log


*   Missing values were present in the raw data and to get rid of those

    missing values, I used Han's 4th method given there was a skew and blanks had to be filled.
*   There were inconsistent categories and it was the same groups but with  different spelling or grammar. I got rid of the multiple groups so that I could compress the data and have only 3 main groups for housing.


*   There were errors and extremes that wouldn't correlate to the real world. Impossible ages were removed and absolute value was applied to hours worked less than zero.

*   Duplicates present were seen in the student_records data. I got rid of duplicate student id's. However, I did not touch course_enrollments or weekly_activity as the duplicate information could belong to multiple unique student id's.

*   Standardizing the key and integrating the datasets ensures that the data can be easier to read especially having it all combined onto one table.
*   To ennsure that the roster was clean, I verified that the rows matched. This allowed me to confirm dataset integrity.


*   Dates matter for enrollment periods. Parsing the enrollment date made sure that every date was recognized and the format was standardized and not lost.



*   Adding the z-score and final_gpa formats the data so that analysis could be made and optimizes student data reporting to the Provost.

# D11 - Payoff

In [ ]:
print(clean_mean_gpa)

2.3053600000000003


Using the clean data, the mean GPA is 2.31. One relationship I find to be interesting is that those who live on campus tend to have a higher GPA than those who live off campus or with family. Cleaning the data made relationships much easier and clearer to see, as the raw data had many mistakes that were hard to understand.